In [0]:
import pyspark
import logging
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DoubleType,DateType,TimestampType

In [0]:
class SilverTransformation:

    def __init__(self, spark, catalog_name):
        self.spark = spark
        self.catalog_name = catalog_name
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
            )
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
    
    def read_bronze(self):
        table_name = f"{self.catalog_name}.bronze.bronze_table"
        self.logger.info(
            f"Reading Bronze table: {table_name}"
        )
        try:
            bronze_df = self.spark.table(table_name)
            self.logger.info(
                "Bronze table read successfully"
            )
            return bronze_df
        except Exception as e:
            raise self.logger.error(
                f"Failed to read Bronze table: {str(e)}")
            
    def drop_columns(self, df):
        self.logger.info("Starting column removal")
        try:
            silver_df = df.drop("ingestion_timestamp", "source_system", "operation_type")
            self.logger.info(f"Dropped columns")
            return silver_df
        
        except Exception as e:
            raise self.logger.error(f"Failed to drop columns: {str(e)}")

    def drop_duplicates(self, df):
        self.logger.info("Starting duplicate removal")
        try:
            silver_df = df.dropDuplicates(["transaction_id"])
            self.logger.info("Duplicates removed using transaction_id")
            return silver_df

        except Exception as e:
            raise self.logger.error(f"Failed to remove duplicates: {str(e)}")

    def create_silver_table(self, df):

        table_name = (f"{self.catalog_name}.silver.silver_table")
        self.logger.info(f"Writing Silver table: {table_name}")
        try:
            (
                df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name)
            )
            self.logger.info(
                f"Silver table created successfully: {table_name}"
            )
            return True
        except Exception as e:
            raise self.logger.error(f"Failed to create Silver table: {str(e)}")

    def run(self):

        self.logger.info("========== Silver transformation started ===========")
        try:
            bronze_df = self.read_bronze()
            silver_df = self.drop_columns(bronze_df)
            silver_df = self.drop_duplicates(silver_df)
            self.create_silver_table(silver_df)
            self.logger.info("========== Silver transformation completed successfully ==========")
            return True

        except Exception as e:
            raise self.logger.error(f"Silver transformation failed: {str(e)}")


In [0]:
CATALOG_NAME = "oag"

silver = SilverTransformation(
    spark=spark,
    catalog_name=CATALOG_NAME
)

silver.run()